In [ ]:
import os, requests, pickle, sklearn
import numpy as np

np.random.seed(1337)  # For reproducibility

BASE_URL = "http://ip:port" # Do not forget to change to your target information

In [103]:
# Write down Starter Code
def get_challenge(phase):
    """Fetch challenge data for specified phase"""
    r = requests.get(f"{BASE_URL}/challenge/{phase}")
    return r.json()

def submit_solutions(phase, solutions):
    """Submit solutions for validation"""
    r = requests.post(
        f"{BASE_URL}/submit/{phase}",
        json={"solutions": solutions}
    )
    return r.json()

In [104]:
def load_model():
    """Download and load the model bundle"""
    # Download model
    r = requests.get(f"{BASE_URL}/model/download")
    with open("model.pkl", "wb") as f:
        f.write(r.content)

    # Load bundle
    with open("model.pkl", "rb") as f:
        bundle = pickle.load(f)

    return bundle

# Examine model structure
bundle = load_model()
print(f"Keys in bundle: {bundle.keys()}")
print(f"Model type: {type(bundle['classifier'])}")
print(f"Number of features: {len(bundle['feature_names'])}")

Keys in bundle: dict_keys(['classifier', 'vectorizer', 'feature_names', 'classes'])
Model type: <class 'sklearn.naive_bayes.MultinomialNB'>
Number of features: 89972


In [105]:
def predict(text):
    """Query prediction API"""
    r = requests.post(
        f"{BASE_URL}/predict",
        json={"text": text}
    )
    return r.json()

# Test baseline probability
sample = "This movie was terrible and boring"
result = predict(sample)
print(f"Label: {result['label']}")
print(f"Positive probability: {result['positive_probability']:.3f}")

Label: negative
Positive probability: 0.002


In [106]:
# Phase 1: White box challenge

# Get challenge information 
whitebox_challenge = get_challenge("whitebox")

reviews = whitebox_challenge["reviews"]

# Assign classifier and vectorizer
classifier = bundle['classifier']
vectorizer = bundle['vectorizer']

# Get feature names and probabilities
feature_names = vectorizer.get_feature_names_out() # Model vocabluary 
positive_log_probs = classifier.feature_log_prob_[1]  # Positive class
negative_log_probs = classifier.feature_log_prob_[0]  # Negative class

print("Model vocabluary contains " + str(len(feature_names)) + " words.")

Model vocabluary contains 89972 words.


In [107]:
# Calculate negativeness scores

negativeness_scores = []
for i, word in enumerate(feature_names):
    positive_prob = np.exp(positive_log_probs[i])
    negative_prob = np.exp(negative_log_probs[i])
    negativeness = negative_prob / (positive_prob + 1e-10)
    negativeness_scores.append((word, negativeness, positive_prob, negative_prob))

In [108]:
# Sort by negativeness
negativeness_scores.sort(key=lambda x: x[1], reverse=True)
top_candidate_words = negativeness_scores[:100]

print(f"[+] Top 10 GoodWords (most 'negative-like'):")
for word, score, hp, sp in top_candidate_words[:10]:
    print(f"    {word:15} | negativeness: {score:8.2f} | positive_prob: {hp*100000:.4f} | negative_prob: {sp*100000:.4f}")


[+] Top 10 GoodWords (most 'negative-like'):
    hours life      | negativeness:    75.02 | positive_prob: 0.0516 | negative_prob: 3.8751
    boll            | negativeness:    73.00 | positive_prob: 0.1033 | negative_prob: 7.5408
    acting horrible | negativeness:    71.98 | positive_prob: 0.0516 | negative_prob: 3.7180
    prom night      | negativeness:    67.92 | positive_prob: 0.0516 | negative_prob: 3.5086
    worst movies seen | negativeness:    61.84 | positive_prob: 0.0516 | negative_prob: 3.1944
    acting awful    | negativeness:    60.83 | positive_prob: 0.0516 | negative_prob: 3.1420
    worst films     | negativeness:    58.80 | positive_prob: 0.1033 | negative_prob: 6.0745
    br br worst     | negativeness:    51.71 | positive_prob: 0.1033 | negative_prob: 5.3414
    br worst        | negativeness:    51.71 | positive_prob: 0.1033 | negative_prob: 5.3414
    uwe             | negativeness:    51.71 | positive_prob: 0.1033 | negative_prob: 5.3414


In [109]:
# Testing
def augment_message(message, words_to_add):
    """Append candidate words to a message"""
    if len(words_to_add) > 0:
        return message + " " + " ".join(words_to_add)
    return message

# Test augmentation on one example using the top 5 words
sample_positive = reviews[0]['text']
sample_augmented = augment_message(
    sample_positive,
    [w for w, _, _, _ in top_candidate_words[:5]]
)
print(f"\nOriginal: {sample_positive}...")
print(f"Augmented: {sample_augmented}...")


Original: I had the great pleasure of recently viewing this beautifully filmed wide-screen adaption of the the 1943 stage revival (which unlike the original 1935 production) which included extensive spoken recitatives. This had been the fashion at the time, so to blame the film for an 16 year tradition. The film should be seen if only for Sammy Davis Jrs brilliant catlike performance as Sportin' Life, creeping in and out of shadows. His seduction of Dorothy Dadridge's BESS "There's a Boat dat's leavin' soon for New York," is one of many highlights. Nearly all of the principal music is intact and beautifully sung. It certainly never bores which the recent PBS and MET versions did. It was a pleasure to see that time had not diminished the movie, and hopefully it will be made available in the near future for the generations that haven't had a chance to experience it....
Augmented: I had the great pleasure of recently viewing this beautifully filmed wide-screen adaption of the the 1943 st

In [110]:
# Define test points from baseline (0) to saturation (30)
word_counts = [0, 5, 10, 15, 20, 25, 30] # We should use up to 30 words to augment rewiev
attack_results = []
whitebox_solutions = []

print(f"[*] Testing word counts: {word_counts}")

for num_words in word_counts:
    # Select the top N candidate words for this iteration
    selected_words = [w for w, _, _, _ in top_candidate_words[:num_words]]

    # Show which words we're using (first iteration only for clarity)
    if num_words == 5:
        print(f"  Using words: {', '.join(selected_words)}")

    # Count how many positive messages considered negative after augmentation
    evaded = 0
    for message in reviews:
        # Augment the message
        augmented = augment_message(message['text'], selected_words)

        # Transform and predict
        vec = vectorizer.transform([augmented])
        prob = classifier.predict_proba(vec)[0]

        # Check evasion: negative probability > positive probability
        if prob[0] > prob[1]:
            evaded += 1
            whitebox_solutions.append({"id": message["id"], "augmented_text": augmented}) # Write down every successful label flips

    # Record results for this configuration
    evasion_rate = (evaded / len(reviews)) * 100
    attack_results.append({
        'num_words': num_words,
        'evasion_rate': evasion_rate,
        'evaded': evaded,
        'total': len(reviews)
    })

    print(f"  Words: {num_words:2d} | Evasion: {evasion_rate:6.2f}% ({evaded}/{len(reviews)})")

# Deduplicate solutions by ID, keeping the first occurrence of each
# We should pass only unique IDs to API
seen = set()
unique_solutions = []
for s in whitebox_solutions:
    if s["id"] not in seen:
        seen.add(s["id"])
        unique_solutions.append(s)

submit_solutions("whitebox", unique_solutions)

[*] Testing word counts: [0, 5, 10, 15, 20, 25, 30]
  Words:  0 | Evasion:   0.00% (0/10)
  Using words: hours life, boll, acting horrible, prom night, worst movies seen
  Words:  5 | Evasion:  80.00% (8/10)
  Words: 10 | Evasion:  90.00% (9/10)
  Words: 15 | Evasion: 100.00% (10/10)
  Words: 20 | Evasion: 100.00% (10/10)
  Words: 25 | Evasion: 100.00% (10/10)
  Words: 30 | Evasion: 100.00% (10/10)


{'message': 'White-box phase complete! Proceed to black-box challenges.',
 'phase_complete': True,
 'results': [{'id': 'wb_0', 'success': True, 'words_added': 10},
  {'id': 'wb_1', 'success': True, 'words_added': 10},
  {'id': 'wb_2', 'success': True, 'words_added': 10},
  {'id': 'wb_3', 'success': True, 'words_added': 10},
  {'id': 'wb_4', 'success': True, 'words_added': 10},
  {'id': 'wb_6', 'success': True, 'words_added': 10},
  {'id': 'wb_7', 'success': True, 'words_added': 10},
  {'id': 'wb_9', 'success': True, 'words_added': 10},
  {'id': 'wb_5', 'success': True, 'words_added': 20},
  {'id': 'wb_8', 'success': True, 'words_added': 30}]}

In [122]:
# Phase 2: Black Box Challenge
blackbox_challenge = get_challenge("blackbox")

# I will use list that have negativenes scores and reverse it to get most negetive words in open model. 
# It will be our foundation to attacking black box model.

negativeness_scores.sort(key=lambda x: x[1])
positive_words = [item[0] for item in negativeness_scores[:80]] # 80 least negative words in an open model

In [ ]:
imp = [] # List that contains words and how it affects on review score
for review in blackbox_challenge["reviews"]:
    for w in positive_words: # Check every positive word
        p_base = predict(review["text"])["positive_probability"] # Positive probability for clear review
        p2 = predict(review["text"] + " " + w)["positive_probability"] # Get positive probability for negative rewiev with added good word
        imp.append((w, p2 - p_base)) # Write results to a list 
imp.sort(key=lambda x: x[1], reverse=True) # Sort words by impact score, highest first
top = imp[:40] 

In [128]:
positive_reviews = [] # List for storing reviews with fliped labels

for review in blackbox_challenge["reviews"]: # Iterate over reviews in challenge
    aug = review["text"]
    for i, w in enumerate(top, 1): # Iterate over candidate words
        if i >= 40:
            break
        aug = aug + " " + w[0] # Append current word to the message
        lab = predict(aug)["label"] # Check if model now classifies it as positive
        if lab == "positive": # Stop early if label already flipped
            positive_reviews.append({"id": review["id"], "augmented_text": aug}) # Store review with fliped label
            break

In [129]:
# Deduplicate solutions by ID, keeping the first occurrence of each
# We should pass only unique IDs to API
seen = set()
unique_solutions = []
for s in positive_reviews:
    if s["id"] not in seen:
        seen.add(s["id"])
        unique_solutions.append(s)

submit_solutions("blackbox", unique_solutions)

{'flag': 'HTB{f34tur3_0bfu5c4t10n_m45t3r3d}',
 'message': 'Congratulations! Both phases complete!',
 'phase_complete': True,
 'results': [{'id': 'bb_0', 'success': True, 'words_added': 7},
  {'id': 'bb_1', 'success': True, 'words_added': 10},
  {'id': 'bb_2', 'success': True, 'words_added': 7},
  {'id': 'bb_3', 'success': True, 'words_added': 4},
  {'id': 'bb_4', 'success': True, 'words_added': 2},
  {'id': 'bb_5', 'success': True, 'words_added': 10},
  {'id': 'bb_6', 'success': True, 'words_added': 12},
  {'id': 'bb_7', 'success': True, 'words_added': 2},
  {'id': 'bb_8', 'success': True, 'words_added': 4},
  {'id': 'bb_9', 'success': True, 'words_added': 2}]}